In [ ]:
import pandas as pd
import numpy as np

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
from experiment_runner import load_dataset, run_single_experiment

np.random.seed(42)

In [ ]:
datasets = ["housing", "adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

#  глобальный таргет (те колонки, которые хотим предсказать моделью)
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

ratios = [0.05, 0.20, 0.50]
mechanisms = ["MCAR", "MAR"]

# число повторов для каждой комбинации (dataset + mechanism + method + params + ratio)
N_REPEATS = 5
BASE_RANDOM_STATE = 42

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
        {"num_strategy": "median"},
    ],
    "KNN numeric": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN gower": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN hybrid": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "MICE": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MICE hybrid": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MissForest": [
        {"n_estimators": 100},
        {"n_estimators": 200},
    ]
}

In [12]:
results_list = []

for dataset_name in datasets:
    df = load_dataset(dataset_name)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    print(f"Dataset {dataset_name} loaded. Shape: {df.shape}")

    columns_with_missing = [list(d.keys())[0] for d in columns_target[dataset_name]]

    for mechanism in mechanisms:

        # кэш пропусков по (ratio, repeat_id), чтобы все методы сравнивались на одинаковых вариантах пропусков
        missing_data_cache = {}
        for ratio in ratios:
            for repeat_id in range(N_REPEATS):
                repeat_seed = BASE_RANDOM_STATE + repeat_id
                df_incomplete, mask = generate_missing_data(
                    data=df,
                    columns_config=columns_target[dataset_name],
                    mechanism=mechanism,
                    ratio=ratio,
                    random_state=repeat_seed
                )
                missing_data_cache[(ratio, repeat_id)] = (df_incomplete, mask)
            print(f"Missing data generated for {dataset_name} | {mechanism} | Ratio: {ratio} | repeats: {N_REPEATS}")

        for method, param_list in EXPERIMENT_CONFIG.items():
            for params in param_list:
                params_str = "_".join([f"{k}={v}" for k, v in params.items()])

                # прогоняем по проценту пропусков и усредняем метрики по повторам
                for ratio in ratios:
                    rmse_values = []
                    acc_values = []
                    time_values = []

                    for repeat_id in range(N_REPEATS):
                        df_incomplete, mask = missing_data_cache[(ratio, repeat_id)]

                        rmse, acc, time_taken = run_single_experiment(
                            df,
                            df_incomplete,
                            mask,
                            method,
                            params,
                            columns_with_missing
                        )

                        if rmse is not None:
                            rmse_values.append(rmse)
                        if acc is not None:
                            acc_values.append(acc)
                        if time_taken is not None:
                            time_values.append(time_taken)

                    rmse_mean = float(np.mean(rmse_values)) if len(rmse_values) > 0 else None
                    rmse_std = float(np.std(rmse_values, ddof=1)) if len(rmse_values) > 1 else 0.0
                    acc_mean = float(np.mean(acc_values)) if len(acc_values) > 0 else None
                    acc_std = float(np.std(acc_values, ddof=1)) if len(acc_values) > 1 else 0.0
                    time_mean = float(np.mean(time_values)) if len(time_values) > 0 else None
                    time_std = float(np.std(time_values, ddof=1)) if len(time_values) > 1 else 0.0

                    # логируем агрегированные результаты локально
                    results_list.append({
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "method": method,
                        "params": params_str,
                        "ratio": ratio,
                        "n_repeats": N_REPEATS,
                        "nrmse": rmse_mean,
                        "accuracy": acc_mean,
                        "imputation_time": time_mean,
                        "nrmse_mean": rmse_mean,
                        "nrmse_std": rmse_std,
                        "accuracy_mean": acc_mean,
                        "accuracy_std": acc_std,
                        "imputation_time_mean": time_mean,
                        "imputation_time_std": time_std
                    })

                print(f"Done: {dataset_name} | {mechanism} | {method} ({params_str})")

# сохраняем все результаты в Excel-таблицу
results_df = pd.DataFrame(results_list)
results_df.to_excel("experiment_results.xlsx", index=False)

Dataset housing loaded. Shape: (5000, 9)
Missing data generated for housing | MCAR | Ratio: 0.05 | repeats: 5
Missing data generated for housing | MCAR | Ratio: 0.2 | repeats: 5
Missing data generated for housing | MCAR | Ratio: 0.5 | repeats: 5
Done: housing | MCAR | Simple (num_strategy=mean)
Done: housing | MCAR | Simple (num_strategy=median)
Done: housing | MCAR | KNN numeric (n_neighbors=3)
Done: housing | MCAR | KNN numeric (n_neighbors=5)
Done: housing | MCAR | KNN numeric (n_neighbors=7)
Done: housing | MCAR | KNN gower (n_neighbors=3)
Done: housing | MCAR | KNN gower (n_neighbors=5)
Done: housing | MCAR | KNN gower (n_neighbors=7)
Done: housing | MCAR | KNN hybrid (n_neighbors=3)
Done: housing | MCAR | KNN hybrid (n_neighbors=5)
Done: housing | MCAR | KNN hybrid (n_neighbors=7)
Done: housing | MCAR | MICE (max_iter=10)
Done: housing | MCAR | MICE (max_iter=50)
Done: housing | MCAR | MICE hybrid (max_iter=10)
Done: housing | MCAR | MICE hybrid (max_iter=50)
Done: housing | MCAR

/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/veronikakamenchuk/Desktop/course_project/missing-data-imputation/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Done: housing | MAR | MICE (max_iter=10)
Done: housing | MAR | MICE (max_iter=50)
Done: housing | MAR | MICE hybrid (max_iter=10)
Done: housing | MAR | MICE hybrid (max_iter=50)
Done: housing | MAR | MissForest (n_estimators=100)
Done: housing | MAR | MissForest (n_estimators=200)
Dataset adult loaded. Shape: (5000, 14)
Missing data generated for adult | MCAR | Ratio: 0.05 | repeats: 5
Missing data generated for adult | MCAR | Ratio: 0.2 | repeats: 5
Missing data generated for adult | MCAR | Ratio: 0.5 | repeats: 5
Done: adult | MCAR | Simple (num_strategy=mean)
Done: adult | MCAR | Simple (num_strategy=median)
Done: adult | MCAR | KNN numeric (n_neighbors=3)
Done: adult | MCAR | KNN numeric (n_neighbors=5)
Done: adult | MCAR | KNN numeric (n_neighbors=7)
Done: adult | MCAR | KNN gower (n_neighbors=3)
Done: adult | MCAR | KNN gower (n_neighbors=5)
Done: adult | MCAR | KNN gower (n_neighbors=7)
Done: adult | MCAR | KNN hybrid (n_neighbors=3)
Done: adult | MCAR | KNN hybrid (n_neighbors=